In [39]:
import scanpy as sc
import pandas as pd
import numpy as np

In [46]:
arthritis_path = './arthritis/rheumatoid_arthritis_data_filtered.h5ad'

In [47]:
arthritis = sc.read_h5ad(arthritis_path)


In [48]:
print(arthritis.obs_keys())
print(len(arthritis.var_names))
print(arthritis.var_names[:10])


['nCount_RNA', 'nFeature_RNA', 'Lane', 'demux_doublet_call', 'demux_RD_TOTL', 'demux_RD_PASS', 'demux_RD_UNIQ', 'demux_N_SNP', 'demux_PRB_DBL', 'percent_mt', 'scrub_doublets', 'batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'norm_library_size', 'S_score', 'G2M_score', 'phase', 'cell_cycle_diff', 'pair_index_CW', 'live_cells', 'dead_cells', 'pct_alive', 'MY_MTX', 'MY_bDMARD', 'MY_pt_global', 'MY_md_global', 'MY_tjc', 'MY_sjc', 'MY_esr', 'MY_crp', 'MY_cdai', 'MY_das28esr4', 'MY_das28crp4', 'MY_RF_status', 'MY_CCP_Status', 'activity_python_crp', 'activity_python_binary_crp', 'activity_python_esr', 'activity_python_binary_esr', 'leiden_r3.0', 'rough_annot', 'fine_annot', 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology

In [16]:
print(arthritis)

AnnData object with n_obs × n_vars = 108717 × 21648
    obs: 'nCount_RNA', 'nFeature_RNA', 'Lane', 'demux_doublet_call', 'demux_RD_TOTL', 'demux_RD_PASS', 'demux_RD_UNIQ', 'demux_N_SNP', 'demux_PRB_DBL', 'percent_mt', 'scrub_doublets', 'batch', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'n_genes', 'n_counts', 'norm_library_size', 'S_score', 'G2M_score', 'phase', 'cell_cycle_diff', 'pair_index_CW', 'live_cells', 'dead_cells', 'pct_alive', 'MY_MTX', 'MY_bDMARD', 'MY_pt_global', 'MY_md_global', 'MY_tjc', 'MY_sjc', 'MY_esr', 'MY_crp', 'MY_cdai', 'MY_das28esr4', 'MY_das28crp4', 'MY_RF_status', 'MY_CCP_Status', 'activity_python_crp', 'activity_python_binary_crp', 'activity_python_esr', 'activity_python_binary_esr', 'leiden_r3.0', 'rough_annot', 'fine_annot', 'tissue_ontology_term_id', 'tissue_type', 'assay_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_t

In [23]:
for col in arthritis.obs.columns:
    unique_values = arthritis.obs[col].nunique()
    if unique_values <= 100 and "clus" not in col.lower():
        print(f"Column: {col}")
        print(arthritis.obs[col].value_counts())
        print("\n")


Column: Lane
Lane
run3-2    10410
run1-3    10383
run1-2    10231
run1-1    10096
run3-4    10024
run3-3     9968
run1-4     9940
run3-1     9649
run2-3     6164
run2-1     5923
run2-2     5779
run2-4     5679
Name: count, dtype: int64


Column: demux_doublet_call
demux_doublet_call
SNG    104246
Name: count, dtype: int64


Column: scrub_doublets
scrub_doublets
False    104214
True         32
Name: count, dtype: int64


Column: batch
batch
1    40650
3    40051
2    23545
Name: count, dtype: int64


Column: total_counts_hb
total_counts_hb
0.0     98541
1.0      5117
2.0       480
3.0        76
4.0        14
6.0         5
5.0         4
7.0         3
9.0         1
12.0        1
8.0         1
10.0        1
15.0        1
17.0        1
Name: count, dtype: int64


Column: phase
phase
G1     69377
G2M    20383
S      14486
Name: count, dtype: int64


Column: pair_index_CW
pair_index_CW
8.0     7967
5.0     7489
23.0    7101
15.0    6873
22.0    6685
10.0    6379
20.0    6116
4.0     6071
19.0

In [24]:
""" 
'label' column을 만든다.
'activity_python_binary_crp' column의 value가 'control'이면 0, 'No'나 'Yes'면 1로 'label'에 매핑하고, 'NA'면 필터링하자. 0과 1은 문자열로 하자
astype('category')로 바꿔주자.
"""

def map_label(value):
    if value == 'control':
        return '0'
    elif value in ['No', 'Yes']:
        return '1'
    else:
        return None
arthritis.obs['label'] = arthritis.obs['activity_python_binary_crp'].apply(map_label)
arthritis = arthritis[~arthritis.obs['label'].isna()]
arthritis.obs['label'] = arthritis.obs['label'].astype('category')


/tmp/ipykernel_1131798/1799875367.py:16: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  arthritis.obs['label'] = arthritis.obs['label'].astype('category')


In [25]:
for col in arthritis.obs.columns:
    unique_values = arthritis.obs[col].nunique()
    if unique_values <= 10 and "clus" not in col.lower():
        print(f"Column: {col}")
        print(arthritis.obs[col].value_counts())
        print("\n")


Column: demux_doublet_call
demux_doublet_call
SNG    104246
Name: count, dtype: int64


Column: scrub_doublets
scrub_doublets
False    104214
True         32
Name: count, dtype: int64


Column: batch
batch
1    40650
3    40051
2    23545
Name: count, dtype: int64


Column: phase
phase
G1     69377
G2M    20383
S      14486
Name: count, dtype: int64


Column: MY_MTX
MY_MTX
0.0    21427
1.0    20566
Name: count, dtype: int64


Column: MY_bDMARD
MY_bDMARD
0.0    41993
1.0     2173
Name: count, dtype: int64


Column: MY_tjc
MY_tjc
1.0    12365
3.0     6239
4.0     5993
2.0     5550
8.0     4628
0.0     4161
9.0     2657
7.0     2573
Name: count, dtype: int64


Column: MY_sjc
MY_sjc
1.0    7645
6.0    6652
2.0    5641
5.0    5113
0.0    4900
9.0    3476
4.0    3383
8.0    2610
7.0    2573
3.0    2173
Name: count, dtype: int64


Column: MY_RF_status
MY_RF_status
Positive    27796
Negative    16370
Name: count, dtype: int64


Column: MY_CCP_Status
MY_CCP_Status
Positive    38553
Negative    

In [32]:
"var names 확인하자. 진짜 gene symbol이 있는지 확인하자."
print(arthritis.var_names[:10])


Index(['ENSG00000186092', 'ENSG00000239945', 'ENSG00000241599',
       'ENSG00000229905', 'ENSG00000237491', 'ENSG00000177757',
       'ENSG00000225880', 'ENSG00000230368', 'ENSG00000187634',
       'ENSG00000188976'],
      dtype='object')


In [38]:
def remap_var_names_to_ppi_symbol_and_drop(
    adata,
    alias_path="/data2/project/bin_jip/Biomarker/data/9606.protein.aliases.v12.0.txt",
    network_path="/data2/project/bin_jip/Biomarker/data/ppi_network.tsv",
):
    # 1) load minimal columns
    alias_df = pd.read_csv(
        alias_path,
        sep="\t",
        usecols=["string_protein_id", "alias"],
        dtype={"string_protein_id": "string", "alias": "string"},
    ).dropna(subset=["string_protein_id", "alias"])

    network_df = pd.read_csv(
        network_path,
        sep="\t",
        usecols=["protein1", "protein2"],
        dtype={"protein1": "string", "protein2": "string"},
    ).dropna(subset=["protein1", "protein2"])

    # normalize
    alias_df["string_protein_id"] = alias_df["string_protein_id"].str.strip()
    alias_df["alias"] = alias_df["alias"].str.strip().str.upper()

    ppi_symbols = pd.Index(network_df["protein1"]).append(pd.Index(network_df["protein2"]))
    ppi_symbols = set(ppi_symbols.astype(str).str.strip().str.upper().tolist())

    # 2) alias -> string_protein_id (deterministic first)
    alias_to_spid = (
        alias_df[["alias", "string_protein_id"]]
        .drop_duplicates(subset="alias", keep="first")
        .set_index("alias")["string_protein_id"]
    )

    # 3) string_protein_id -> gene symbol (must exist in PPI symbols)
    spid_to_symbol = (
        alias_df.loc[alias_df["alias"].isin(ppi_symbols), ["string_protein_id", "alias"]]
        .sort_values(["string_protein_id", "alias"])
        .drop_duplicates(subset="string_protein_id", keep="first")
        .set_index("string_protein_id")["alias"]
    )

    # 4) vectorized mapping for var_names
    var_orig = pd.Series(adata.var_names.astype(str), index=np.arange(adata.n_vars), name="var_name_original")
    var_norm = var_orig.str.strip().str.upper()

    spid_series = var_norm.map(alias_to_spid)
    symbol_series = spid_series.map(spid_to_symbol)

    keep_mask = symbol_series.notna().to_numpy()
    removed_count = int((~keep_mask).sum())
    mapped_count = int(keep_mask.sum())

    # 5) drop unmapped vars + replace var_names
    adata_new = adata[:, keep_mask].copy()
    adata_new.var["var_name_original"] = var_orig[keep_mask].to_numpy()
    adata_new.var["string_protein_id"] = spid_series[keep_mask].to_numpy()
    adata_new.var_names = pd.Index(symbol_series[keep_mask].to_numpy())

    dup_before_unique = int(adata_new.var_names.duplicated().sum())
    if dup_before_unique > 0:
        adata_new.var_names_make_unique()

    final_ppi_overlap = len(set(adata_new.var_names.astype(str).str.upper()) & ppi_symbols)

    print(f"[mapping] original n_vars: {adata.n_vars}")
    print(f"[mapping] mapped: {mapped_count}, removed: {removed_count}")
    print(f"[mapping] duplicated symbols before make_unique: {dup_before_unique}")
    print(f"[mapping] final n_vars: {adata_new.n_vars}")
    print(f"[mapping] final var_names overlap with PPI symbols: {final_ppi_overlap}")

    report = pd.DataFrame(
        {
            "var_name_original": var_orig,
            "string_protein_id": spid_series,
            "mapped_symbol": symbol_series,
            "kept": symbol_series.notna(),
        }
    )
    return adata_new, report

arthritis_mapped, map_report = remap_var_names_to_ppi_symbol_and_drop(arthritis)
map_report.to_csv("/data2/project/bin_jip/Biomarker/data/arthritis/varname_mapping_report.tsv", sep="\t", index=False)
arthritis_mapped.write_h5ad("/data2/project/bin_jip/Biomarker/data/arthritis/rheumatoid_arthritis_data_filtered.h5ad")


[mapping] original n_vars: 21648
[mapping] mapped: 13046, removed: 8602
[mapping] duplicated symbols before make_unique: 480
[mapping] final n_vars: 13046
[mapping] final var_names overlap with PPI symbols: 12566


In [26]:
"""
'donor_id' column in arthitis has k unique values.
k= ?
"""
for col in arthritis.obs.columns:
    unique_values = arthritis.obs[col].nunique()
    print(f"Column: {col} (Unique values: {unique_values})")

"""
각 label 당 patient 수는 다음과 같다.
"""

label_counts = arthritis.obs.groupby('label')['donor_id'].nunique()
print(label_counts)


Column: nCount_RNA (Unique values: 1883)
Column: nFeature_RNA (Unique values: 868)
Column: Lane (Unique values: 12)
Column: demux_doublet_call (Unique values: 1)
Column: demux_RD_TOTL (Unique values: 26719)
Column: demux_RD_PASS (Unique values: 720)
Column: demux_RD_UNIQ (Unique values: 382)
Column: demux_N_SNP (Unique values: 284)
Column: demux_PRB_DBL (Unique values: 8510)
Column: percent_mt (Unique values: 44797)
Column: scrub_doublets (Unique values: 2)
Column: batch (Unique values: 3)
Column: n_genes_by_counts (Unique values: 868)
Column: total_counts (Unique values: 1883)
Column: total_counts_mt (Unique values: 359)
Column: pct_counts_mt (Unique values: 38125)
Column: total_counts_ribo (Unique values: 1045)
Column: pct_counts_ribo (Unique values: 45706)
Column: total_counts_hb (Unique values: 14)
Column: pct_counts_hb (Unique values: 1899)
Column: n_genes (Unique values: 868)
Column: n_counts (Unique values: 1883)
Column: norm_library_size (Unique values: 1852)
Column: S_score (U

/tmp/ipykernel_1131798/3764037300.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  label_counts = arthritis.obs.groupby('label')['donor_id'].nunique()


In [28]:

arthritis.write_h5ad('./arthritis/rheumatoid_arthritis_data_mapped.h5ad')